# Лабораторная работа №18: Графы знаний (Knowledge Graphs) и RAG

**ФИО студента:** _______________________
**Группа:** _______________________

## Цель работы
Изучить технологию графов знаний (Knowledge Graphs) и их применение в RAG-системах. Научиться извлекать сущности и связи из текста, строить графы и использовать их для улучшения качества ответов на вопросы о связях между объектами.

## Теоретические сведения
### Что такое граф знаний?
Граф знаний — это структура данных, представляющая знания в виде сети сущностей (узлов) и отношений между ними (рёбер). В отличие от векторного поиска, графы позволяют явно моделировать связи и выполнять логический вывод.

### Компоненты графа
- **Сущности (Entities):** Объекты реального мира (люди, организации, места)
- **Отношения (Relations):** Связи между сущностями (работает_в, находится_в, знает)
- **Свойства (Properties):** Атрибуты сущностей (возраст, дата рождения)

### Применение в RAG
1. **Улучшение контекста:** Граф предоставляет структурированный контекст о связях
2. **Многошаговые запросы:** Возможность отвечать на вопросы типа "Как связаны Х и У?"
3. **Объяснимость:** Путь в графе показывает логику ответа

## Задание
### Уровень 1 (Базовый)
1. Извлеките сущности и связи из небольшого текста (5-10 предложений)
2. Постройте граф с помощью NetworkX
3. Визуализируйте граф

### Уровень 2 (Продвинутый)
1. Реализуйте функцию поиска кратчайшего пути между двумя сущностями
2. Создайте систему вопросов-ответов на основе обхода графа
3. Объедините векторный поиск и графовый подход

### Уровень 3 (Индивидуальный)
1. Реализуйте сохранение графа в базу данных (Neo4j или JSON)
2. Добавьте поддержку взвешенных рёбер (сила связи)
3. Создайте гибридный RAG с приоритетом графовых данных для вопросов о связях

## Ход работы

### Шаг 1: Установка зависимостей

In [ ]:
%pip install langchain langchain-openai networkx matplotlib spacy -q
%python -m spacy download ru_core_news_sm

### Шаг 2: Загрузка тестового текста

In [ ]:
text = """
Иван Петров работает в компании TechCorp на позиции старшего разработчика. 
TechCorp расположена в Москве и специализируется на искусственном интеллекте. 
Иван окончил МГУ и ранее работал в Яндекс. 
Его коллега Мария Сидорова работает менеджером проектов в том же офисе. 
Мария окончила СПбГУ и переехала в Москву из Санкт-Петербурга. 
TechCorp недавно заключила партнёрство с компанией AI Solutions, которая находится в Казани.
Директор TechCorp - Алексей Волков, который также является выпускником МГУ.
"""

print(text)

### Шаг 3: Извлечение сущностей и связей с помощью LLM

In [ ]:
import json
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

# os.environ["OPENAI_API_KEY"] = "your-key-here"
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", """Ты эксперт по извлечению знаний. Извлеки все сущности и связи из текста.
    Верни результат в формате JSON:
    {
      "entities": [{"name": "имя", "type": "тип"}],
      "relations": [{"from": "сущность1", "to": "сущность2", "relation": "связь"}]
    }
    Типы сущностей: PERSON, ORGANIZATION, LOCATION, POSITION, UNIVERSITY
    Возвращай ТОЛЬКО JSON без дополнительного текста.
    """),
    ("human", "{text}")
])

extraction_chain = extraction_prompt | llm

In [ ]:
# Извлечение (для демонстрации используем предопределённые данные)
# response = extraction_chain.invoke({"text": text})
# kg_data = json.loads(response.content)

# Предопределённые данные для примера
kg_data = {
    "entities": [
        {"name": "Иван Петров", "type": "PERSON"},
        {"name": "TechCorp", "type": "ORGANIZATION"},
        {"name": "Москва", "type": "LOCATION"},
        {"name": "МГУ", "type": "UNIVERSITY"},
        {"name": "Яндекс", "type": "ORGANIZATION"},
        {"name": "Мария Сидорова", "type": "PERSON"},
        {"name": "СПбГУ", "type": "UNIVERSITY"},
        {"name": "Санкт-Петербург", "type": "LOCATION"},
        {"name": "AI Solutions", "type": "ORGANIZATION"},
        {"name": "Казань", "type": "LOCATION"},
        {"name": "Алексей Волков", "type": "PERSON"}
    ],
    "relations": [
        {"from": "Иван Петров", "to": "TechCorp", "relation": "работает_в"},
        {"from": "TechCorp", "to": "Москва", "relation": "расположена_в"},
        {"from": "Иван Петров", "to": "МГУ", "relation": "окончил"},
        {"from": "Иван Петров", "to": "Яндекс", "relation": "ранее_работал_в"},
        {"from": "Мария Сидорова", "to": "TechCorp", "relation": "работает_в"},
        {"from": "Мария Сидорова", "to": "СПбГУ", "relation": "окончила"},
        {"from": "Мария Сидорова", "to": "Санкт-Петербург", "relation": "переехала_из"},
        {"from": "TechCorp", "to": "AI Solutions", "relation": "партнёрство_с"},
        {"from": "AI Solutions", "to": "Казань", "relation": "находится_в"},
        {"from": "Алексей Волков", "to": "TechCorp", "relation": "директор"},
        {"from": "Алексей Волков", "to": "МГУ", "relation": "окончил"}
    ]
}

print(f"Найдено сущностей: {len(kg_data['entities'])}")
print(f"Найдено связей: {len(kg_data['relations'])}")

### Шаг 4: Построение графа

In [ ]:
import networkx as nx

# Создание графа
G = nx.DiGraph()

# Добавление узлов
for entity in kg_data['entities']:
    G.add_node(entity['name'], type=entity['type'])

# Добавление рёбер
for relation in kg_data['relations']:
    G.add_edge(relation['from'], relation['to'], relation=relation['relation'])

print(f"Граф создан: {G.number_of_nodes()} узлов, {G.number_of_edges()} рёбер")

### Шаг 5: Визуализация графа

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 10))
pos = nx.spring_layout(G, k=2, iterations=50)

# Цвета для разных типов сущностей
color_map = {
    'PERSON': '#FF6B6B',
    'ORGANIZATION': '#4ECDC4',
    'LOCATION': '#FFE66D',
    'UNIVERSITY': '#95E1D3'
}

node_colors = [color_map.get(G.nodes[node]['type'], '#gray') for node in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=3000, alpha=0.8)
nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=20)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold')

# Подписи рёбер
edge_labels = {(u, v): d['relation'] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7)

plt.title("Граф знаний", fontsize=16)
plt.axis('off')
plt.tight_layout()
plt.show()

### Шаг 6: Поиск путей в графе

In [ ]:
def find_connection_path(entity1, entity2):
    """Находит путь между двумя сущностями"""
    try:
        path = nx.shortest_path(G, source=entity1, target=entity2)
        
        # Формирование описания пути
        description = []
        for i in range(len(path) - 1):
            rel = G[path[i]][path[i+1]]['relation']
            description.append(f"{path[i]} --[{rel}]--> {path[i+1]}")
        
        return {
            "path": path,
            "description": " -> ".join(description),
            "length": len(path) - 1
        }
    except nx.NetworkXNoPath:
        return {"error": f"Нет связи между {entity1} и {entity2}"}

# Примеры
print("Связь между Иван Петров и Мария Сидорова:")
result = find_connection_path("Иван Петров", "Мария Сидорова")
print(result)

print("\nСвязь между МГУ и AI Solutions:")
result = find_connection_path("МГУ", "AI Solutions")
print(result)

### Шаг 7: Ответы на вопросы с использованием графа

In [ ]:
def answer_question_from_graph(question):
    """Простая система ответов на основе графа"""
    question_lower = question.lower()
    
    # Поиск связей
    if "как связан" in question_lower or "как связаны" in question_lower:
        # Извлечение имён из вопроса (упрощённо)
        entities = [e['name'] for e in kg_data['entities'] if e['name'] in question]
        if len(entities) >= 2:
            return find_connection_path(entities[0], entities[1])
    
    # Поиск соседей
    if "где работает" in question_lower:
        for entity in kg_data['entities']:
            if entity['name'] in question and entity['type'] == 'PERSON':
                neighbors = list(G.successors(entity['name']))
                works_at = [n for n in neighbors if G[entity['name']][n]['relation'] == 'работает_в']
                if works_at:
                    return f"{entity['name']} работает в {works_at[0]}"
    
    return "Не могу найти ответ в графе знаний"

# Тестирование
questions = [
    "Как связан Иван Петров и Мария Сидорова?",
    "Где работает Иван Петров?",
    "Как связаны МГУ и Алексей Волков?"
]

for q in questions:
    print(f"\nВопрос: {q}")
    print(f"Ответ: {answer_question_from_graph(q)}")

## Контрольные вопросы
1. В чём преимущества графов знаний перед векторным поиском?
2. Какие типы вопросов лучше всего решаются с помощью графов?
3. Как можно комбинировать графовый и векторный подходы в RAG?
4. Какие проблемы могут возникнуть при извлечении сущностей из текста?

## Выводы
В работе изучены основы построения и использования графов знаний. Реализована система извлечения сущностей, построения графа и поиска связей. Показано, как графы могут улучшать ответы на вопросы о взаимоотношениях между объектами.